# Graph Neural Networks: Visual Message Passing and a Real Comparison

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/yfeng-hsm/KI_Geodatenanalyse_SS26/blob/main/lectures/02_deep_learning/notebooks/gnn_visual_message_passing_colab.ipynb)

This tutorial is designed as the GNN part of the deep learning lecture sequence:

1. neural networks and CNNs,
2. transformers,
3. graph neural networks.

The notebook has two goals:

- **Part A:** make one GNN update visible step by step. Students can see node features, edge weights, learnable weights, messages, aggregation, and updated node states.
- **Part B:** use a real graph dataset to compare a feature-only machine learning baseline with a GNN. The point is to make the benefit of neighbourhood information visible.

The code uses PyTorch and PyTorch Geometric so that the conceptual explanation is connected to standard GNN tooling.

## 0. Colab Setup

PyTorch is usually preinstalled in Colab. PyTorch Geometric is installed here because it provides standard graph datasets and layers.

In [ ]:
!pip -q install torch-geometric networkx matplotlib pandas scikit-learn ipywidgets pyvis

In [ ]:
import math
import random
import tempfile
from pathlib import Path

import ipywidgets as widgets
import matplotlib.pyplot as plt
import networkx as nx
import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
from IPython.display import HTML, IFrame, Math, clear_output, display
from pyvis.network import Network
from torch import nn
from torch_geometric.datasets import Planetoid
from torch_geometric.nn import GCNConv
from torch_geometric.utils import k_hop_subgraph

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
DEVICE

# Part A: A GNN Update, Made Visible

A basic graph neural network layer updates each node by mixing information from its neighbours.

For a GCN-style layer, one update can be written as:

\[
H^{(1)} = \sigma\left(\hat{A} X W\right)
\]

where:

- \(X\) is the input node feature matrix,
- \(W\) is a learnable weight matrix,
- \(\hat{A}\) is the normalized adjacency matrix with self-loops,
- \(\sigma\) is a nonlinear activation function,
- \(H^{(1)}\) is the updated node representation.

The graph is tiny so every number can be inspected. The next interactive cell is the most important one: click **Update one layer** repeatedly and watch how values and colors change.

In [ ]:
display(Math(r"H^{(k+1)} = \mathrm{ReLU}\left(\hat{A} H^{(k)} W^{(k)}\right)"))
display(Math(r"\hat{A}_{ij} = \frac{1}{\sqrt{d_i d_j}}"))

In [ ]:
toy_edges = [
    (0, 1), (0, 2),
    (1, 2), (1, 3),
    (2, 4),
    (3, 4), (3, 5),
    (4, 5),
]

node_names = {
    0: "A: river",
    1: "B: river",
    2: "C: mixed",
    3: "D: street",
    4: "E: street",
    5: "F: street",
}

X0 = torch.tensor([
    [1.00, 0.10],
    [0.90, 0.20],
    [0.55, 0.55],
    [0.20, 0.90],
    [0.10, 1.00],
    [0.05, 0.95],
], dtype=torch.float32)

W0 = torch.tensor([
    [ 1.20, -0.70],
    [-0.35,  1.10],
], dtype=torch.float32)

G_toy = nx.Graph()
G_toy.add_edges_from(toy_edges)
pos_toy = nx.spring_layout(G_toy, seed=SEED)

In [ ]:
def normalized_adjacency(num_nodes, edges):
    A = torch.zeros((num_nodes, num_nodes), dtype=torch.float32)
    for i, j in edges:
        A[i, j] = 1
        A[j, i] = 1
    A = A + torch.eye(num_nodes)
    degree = A.sum(dim=1)
    d_inv_sqrt = torch.diag(torch.pow(degree, -0.5))
    A_hat = d_inv_sqrt @ A @ d_inv_sqrt
    return A, A_hat, degree


def feature_color(values):
    values = values.detach().cpu().numpy()
    return values[:, 0] - values[:, 1]


def draw_graph_state(features, title, edge_weights=None, cmap="coolwarm"):
    fig, ax = plt.subplots(figsize=(7, 5))
    scalar = feature_color(features)
    nodes = nx.draw_networkx_nodes(
        G_toy,
        pos_toy,
        node_color=scalar,
        cmap=cmap,
        vmin=-1.2,
        vmax=1.2,
        node_size=1150,
        edgecolors="#222",
        linewidths=1.2,
        ax=ax,
    )
    nx.draw_networkx_edges(G_toy, pos_toy, width=1.8, alpha=0.55, ax=ax)
    labels = {i: f"{i}\\n[{features[i,0]:.2f}, {features[i,1]:.2f}]" for i in G_toy.nodes}
    nx.draw_networkx_labels(G_toy, pos_toy, labels=labels, font_size=9, ax=ax)
    if edge_weights:
        nx.draw_networkx_edge_labels(G_toy, pos_toy, edge_labels=edge_weights, font_size=8, ax=ax)
    fig.colorbar(nodes, ax=ax, shrink=0.75, label="feature[0] - feature[1]")
    ax.set_title(title)
    ax.axis("off")
    plt.show()


def feature_table(features, title):
    df = pd.DataFrame(features.detach().cpu().numpy(), columns=["feature_0", "feature_1"])
    df.insert(0, "node", [node_names[i] for i in range(len(df))])
    print(title)
    display(df.round(3))


A_with_self_loops, A_hat, degree = normalized_adjacency(num_nodes=X0.size(0), edges=toy_edges)
feature_table(X0, "Initial node features X")
draw_graph_state(X0, "Initial node features")

## Interactive Message Passing App with PyVis

This app is meant to be used before the static step-by-step cells below.

- Choose a target node.
- The graph is rendered with **PyVis**, so nodes can be dragged, zoomed, and inspected interactively.
- Hover over a node to see its current state and label.
- Red arrows point from source nodes into the selected target node.
- The animated message-flow strip below the graph shows messages moving along those arrows; the self-loop is shown as a pulsing circle around the target.
- Click **Update one layer** to apply one GCN-style update.
- The right-hand explanation panel shows the formula, weight matrix, incoming messages, and target-node result.

The graph state is \(H^{(k)}\). Node color represents `feature_0 - feature_1`, node labels show the current two-dimensional state, and arrows/animations show the message passing process.

In [ ]:
APP_WEIGHTS = [
    W0,
    torch.tensor([[0.80, 0.35], [0.25, 0.90]], dtype=torch.float32),
    torch.tensor([[0.65, -0.15], [0.20, 0.75]], dtype=torch.float32),
]

app_state = {
    "layer": 0,
    "features": X0.clone(),
    "last_update": None,
}

target_dropdown = widgets.Dropdown(
    options=[(f"{i}: {node_names[i]}", i) for i in range(X0.size(0))],
    value=3,
    description="Target",
)
update_button = widgets.Button(description="Update one layer", button_style="primary", icon="refresh")
reset_button = widgets.Button(description="Reset", button_style="", icon="undo")
app_output = widgets.Output()


def message_sources(target_node):
    return [int(src) for src in range(X0.size(0)) if A_hat[target_node, src] > 0]


def message_table_for_target(features, weight, target_node):
    transformed = features @ weight
    rows = []
    for source_node in message_sources(target_node):
        coeff = A_hat[target_node, source_node].item()
        contribution = coeff * transformed[source_node]
        rows.append({
            "source": f"{source_node}: {node_names[source_node]}",
            "edge weight": coeff,
            "after W [0]": transformed[source_node, 0].item(),
            "after W [1]": transformed[source_node, 1].item(),
            "message [0]": contribution[0].item(),
            "message [1]": contribution[1].item(),
        })
    return pd.DataFrame(rows)


def color_for_node(features, node):
    score = float(features[node, 0] - features[node, 1])
    score = max(-1.2, min(1.2, score))
    if score >= 0:
        intensity = int(235 - 75 * min(score / 1.2, 1.0))
        return f"#dc{intensity:02x}{intensity:02x}"
    intensity = int(235 - 75 * min(abs(score) / 1.2, 1.0))
    return f"#{intensity:02x}{intensity:02x}e1"


def pyvis_graph_html(features, target_node, layer):
    neighbours = set(message_sources(target_node))
    net = Network(height="430px", width="100%", directed=True, notebook=True, cdn_resources="in_line", bgcolor="#ffffff")
    net.toggle_physics(False)
    net.set_options("""
    {
      "interaction": {"hover": true, "navigationButtons": true, "keyboard": true},
      "nodes": {"shape": "dot", "font": {"size": 15, "face": "arial"}},
      "edges": {"smooth": false, "font": {"size": 12, "align": "middle"}, "arrows": {"to": {"enabled": false}}}
    }
    """)

    fixed_pos = {
        0: (-240, -150),
        1: (-60, -170),
        2: (-130, -20),
        3: (90, -50),
        4: (-10, 130),
        5: (190, 120),
    }

    for node in range(features.size(0)):
        is_target = node == target_node
        in_messages = node in neighbours
        border = "#b00020" if is_target else ("#333333" if in_messages else "#666666")
        size = 32 if is_target else (27 if in_messages else 24)
        title = (
            f"<b>{node}: {node_names[node]}</b><br>"
            f"layer: H^({layer})<br>"
            f"state: [{features[node,0]:.4f}, {features[node,1]:.4f}]<br>"
            f"feature_0 - feature_1: {float(features[node,0] - features[node,1]):.4f}"
        )
        label = f"{node}\n[{features[node,0]:.2f}, {features[node,1]:.2f}]"
        x, y = fixed_pos[node]
        net.add_node(
            node,
            label=label,
            title=title,
            color={"background": color_for_node(features, node), "border": border},
            borderWidth=4 if is_target else 2,
            size=size,
            x=x,
            y=y,
            physics=False,
        )

    active_pairs = set()
    for src in neighbours:
        if src != target_node:
            active_pairs.add(tuple(sorted((src, target_node))))

    for i, j in toy_edges:
        pair = tuple(sorted((i, j)))
        if pair in active_pairs:
            src = j if i == target_node else i
            dst = target_node
            weight = A_hat[dst, src].item()
            net.add_edge(
                src,
                dst,
                label=f"{weight:.2f}",
                title=f"message {src} → {dst}; normalized edge weight: {weight:.4f}",
                color="#d62728",
                width=4,
                arrows="to",
            )
        else:
            weight = A_hat[i, j].item()
            net.add_edge(
                i,
                j,
                label="",
                title=f"normalized edge weight: {weight:.4f}",
                color="#b6bbc2",
                width=1.5,
                arrows="",
            )

    # Self-message is part of A_hat but is not visible as a normal graph edge.
    self_weight = A_hat[target_node, target_node].item()
    net.add_edge(
        target_node,
        target_node,
        label=f"self {self_weight:.2f}",
        title=f"self-loop message; normalized weight: {self_weight:.4f}",
        color="#d62728",
        width=3,
        arrows="to",
    )

    html = net.generate_html(notebook=True)
    legend = """
    <div style='font-family: Arial, sans-serif; font-size: 12px; margin: 6px 0 10px 0; color: #444;'>
      <b>PyVis graph:</b> drag nodes, zoom, and hover for details. Red arrows = messages into target. Self-loop = target keeps its own state. Node color = feature_0 - feature_1.
    </div>
    """
    return html + legend


def message_flow_animation_html(target_node):
    # Coordinates match the PyVis layout, shifted into a compact SVG viewport.
    fixed_pos = {
        0: (90, 65),
        1: (220, 55),
        2: (170, 165),
        3: (330, 140),
        4: (255, 260),
        5: (405, 255),
    }
    tx, ty = fixed_pos[target_node]
    parts = []
    for idx, src in enumerate(message_sources(target_node)):
        sx, sy = fixed_pos[src]
        delay = 0.18 * idx
        if src == target_node:
            parts.append(f"""
            <circle cx='{tx}' cy='{ty}' r='39' fill='none' stroke='#d62728' stroke-width='3' opacity='0.0'>
              <animate attributeName='r' values='32;48;32' dur='1.4s' begin='{delay}s' repeatCount='indefinite'/>
              <animate attributeName='opacity' values='0.1;0.9;0.1' dur='1.4s' begin='{delay}s' repeatCount='indefinite'/>
            </circle>
            """)
        else:
            parts.append(f"""
            <line x1='{sx}' y1='{sy}' x2='{tx}' y2='{ty}' stroke='#d62728' stroke-width='2.5' opacity='0.45' marker-end='url(#arrowhead)'/>
            <circle r='6' fill='#d62728'>
              <animateMotion dur='1.4s' begin='{delay}s' repeatCount='indefinite' path='M {sx},{sy} L {tx},{ty}' />
            </circle>
            """)

    node_parts = []
    for node, (x, y) in fixed_pos.items():
        fill = '#fff4f4' if node == target_node else '#f7f7f7'
        stroke = '#b00020' if node == target_node else '#777'
        node_parts.append(f"<circle cx='{x}' cy='{y}' r='20' fill='{fill}' stroke='{stroke}' stroke-width='2'/><text x='{x}' y='{y+4}' text-anchor='middle' font-size='12' font-weight='700'>{node}</text>")

    return f"""
    <div style='border:1px solid #ddd; border-radius:8px; padding:8px; background:#fff; margin-top:8px;'>
      <div style='font-weight:700; margin-bottom:4px;'>Animated message flow into target node {target_node}</div>
      <svg viewBox='50 20 410 280' width='100%' height='230'>
        <defs>
          <marker id='arrowhead' markerWidth='10' markerHeight='8' refX='9' refY='4' orient='auto'>
            <path d='M0,0 L10,4 L0,8 z' fill='#d62728'></path>
          </marker>
        </defs>
        {''.join(parts)}
        {''.join(node_parts)}
      </svg>
    </div>
    """


def table_html(df, caption):
    return df.round(3).to_html(index=False, classes='gnn-table', border=0, escape=False, caption=caption)


def weight_table_html(weight, layer):
    df = pd.DataFrame(weight.numpy(), index=["input_0", "input_1"], columns=["hidden_0", "hidden_1"]).round(3)
    return df.to_html(classes='gnn-table', border=0, caption=f"Weight matrix W^({layer})")


def render_message_passing_app():
    with app_output:
        clear_output(wait=True)
        layer = app_state["layer"]
        features = app_state["features"]
        target = target_dropdown.value
        weight = APP_WEIGHTS[min(layer, len(APP_WEIGHTS) - 1)]
        transformed = features @ weight
        pre_update = A_hat @ transformed
        post_update = torch.relu(pre_update)
        msg_df = message_table_for_target(features, weight, target)
        just_updated = app_state.get("last_update")

        target_result = pd.DataFrame({
            "pre_relu_0": [pre_update[target, 0].item()],
            "pre_relu_1": [pre_update[target, 1].item()],
            "updated_0": [post_update[target, 0].item()],
            "updated_1": [post_update[target, 1].item()],
        })
        status_text = "This panel previews the next update for the selected target."
        if just_updated is not None:
            status_text = f"Last click applied layer {just_updated['from_layer']} → {just_updated['to_layer']} for target node {just_updated['target']}. The graph now shows H^({layer})."

        explanation = f"""
        <style>
          .gnn-card {{ border: 1px solid #ddd; border-radius: 8px; padding: 12px; background: #fff; height: 700px; overflow-y: auto; }}
          .gnn-title {{ font-size: 16px; font-weight: 700; margin-bottom: 8px; }}
          .formula {{ font-family: ui-monospace, SFMono-Regular, Menlo, monospace; background: #f6f8fa; padding: 8px; border-radius: 6px; margin-bottom: 10px; }}
          .status {{ background:#fff8e1; border-left:4px solid #f2b01e; padding:8px; margin-bottom:10px; }}
          table.gnn-table {{ border-collapse: collapse; width: 100%; margin: 8px 0 14px 0; font-size: 12px; }}
          table.gnn-table caption {{ caption-side: top; text-align: left; font-weight: 700; margin-bottom: 4px; }}
          table.gnn-table th, table.gnn-table td {{ border-bottom: 1px solid #e5e5e5; padding: 4px 6px; text-align: right; }}
          table.gnn-table th:first-child, table.gnn-table td:first-child {{ text-align: left; }}
        </style>
        <div class='gnn-card'>
          <div class='gnn-title'>Operation Explanation</div>
          <div class='status'>{status_text}</div>
          <div class='formula'>H<sup>({layer + 1})</sup> = ReLU(A_hat H<sup>({layer})</sup> W<sup>({layer})</sup>)</div>
          <p><b>Current layer:</b> {layer}</p>
          <p><b>Selected target:</b> {target}: {node_names[target]}</p>
          <p>The red arrows and animated dots show messages that flow into the selected target. Each message is multiplied by the normalized edge weight before aggregation.</p>
          {weight_table_html(weight, layer)}
          {table_html(msg_df, 'Messages into selected target node')}
          {table_html(target_result, 'Selected target node if you apply the next update')}
        </div>
        """

        graph_block = pyvis_graph_html(features, target, layer) + message_flow_animation_html(target)
        left = widgets.HTML(value=graph_block, layout=widgets.Layout(width="58%"))
        right = widgets.HTML(value=explanation, layout=widgets.Layout(width="42%"))
        display(widgets.HBox([left, right], layout=widgets.Layout(width="100%", align_items="flex-start")))


def update_one_layer(_):
    layer = app_state["layer"]
    target = target_dropdown.value
    weight = APP_WEIGHTS[min(layer, len(APP_WEIGHTS) - 1)]
    app_state["features"] = torch.relu(A_hat @ (app_state["features"] @ weight))
    app_state["layer"] = layer + 1
    app_state["last_update"] = {"from_layer": layer, "to_layer": layer + 1, "target": target}
    render_message_passing_app()


def reset_app(_):
    app_state["layer"] = 0
    app_state["features"] = X0.clone()
    app_state["last_update"] = None
    render_message_passing_app()


def target_changed(_):
    app_state["last_update"] = None
    render_message_passing_app()


update_button.on_click(update_one_layer)
reset_button.on_click(reset_app)
target_dropdown.observe(target_changed, names="value")

controls = widgets.HBox([target_dropdown, update_button, reset_button])
display(widgets.VBox([controls, app_output]))
render_message_passing_app()

## Step 1: Add Self-Loops and Normalize Edges

A GCN keeps a node's own information through a self-loop and normalizes messages by node degree:

\[
\hat{A}_{ij} = \frac{1}{\sqrt{d_i d_j}}
\]

This prevents high-degree nodes from dominating simply because they have many edges.

In [ ]:
display(pd.DataFrame(A_with_self_loops.numpy()).astype(int).style.set_caption("Adjacency with self-loops"))
display(pd.DataFrame(A_hat.numpy()).round(3).style.set_caption("Normalized adjacency A_hat"))

edge_weight_labels = {}
for i, j in G_toy.edges:
    edge_weight_labels[(i, j)] = f"{A_hat[i, j]:.2f}"

draw_graph_state(X0, "Graph with normalized edge weights", edge_weights=edge_weight_labels)

## Step 2: Apply Learnable Weights

Before aggregation, every node feature vector is transformed by the learnable weight matrix `W`. This is the neural-network part of the layer.

In [ ]:
display(pd.DataFrame(W0.numpy(), index=["input_feature_0", "input_feature_1"], columns=["hidden_0", "hidden_1"]).round(3).style.set_caption("Weight matrix W"))

XW = X0 @ W0
feature_table(XW, "Transformed node features X @ W")
draw_graph_state(XW, "After applying learnable weights: X @ W")

## Step 3: Aggregate Neighbour Messages

Now each node receives a weighted sum of transformed features from itself and its neighbours. This is where a GNN differs from a standard MLP.

In [ ]:
H_pre = A_hat @ XW
H1 = torch.relu(H_pre)

feature_table(H_pre, "Before ReLU: A_hat @ X @ W")
feature_table(H1, "After ReLU: H1")

draw_graph_state(H1, "Updated node states after one GCN-style layer")

In [ ]:
def explain_node_update(target_node):
    rows = []
    for source_node in range(X0.size(0)):
        coeff = A_hat[target_node, source_node].item()
        if coeff == 0:
            continue
        transformed = XW[source_node]
        contribution = coeff * transformed
        rows.append({
            "target": node_names[target_node],
            "source": node_names[source_node],
            "normalized_weight": coeff,
            "source_hidden_0": transformed[0].item(),
            "source_hidden_1": transformed[1].item(),
            "message_0": contribution[0].item(),
            "message_1": contribution[1].item(),
        })
    df = pd.DataFrame(rows)
    display(df.round(3))
    print("sum of messages:", H_pre[target_node].detach().numpy().round(3))
    print("after ReLU:", H1[target_node].detach().numpy().round(3))


explain_node_update(target_node=3)

## Step 4: Stack Another Layer

A second GNN layer lets information move two hops. The visualization below shows how representations become smoother across connected neighbourhoods.

In [ ]:
W1 = torch.tensor([
    [0.80, 0.35],
    [0.25, 0.90],
], dtype=torch.float32)

H2_pre = A_hat @ (H1 @ W1)
H2 = torch.relu(H2_pre)

feature_table(H2, "After two GCN-style layers")
draw_graph_state(H2, "Updated node states after two layers")

# Part B: Real Data Comparison - Feature-Only ML vs GNN

The goal is to make a clear teaching comparison:

- **Feature-only MLP:** sees each node's feature vector, but ignores graph edges.
- **GCN:** sees the same node features and additionally uses the graph.

On citation networks such as Cora, neighbouring papers often have related topics. A GNN can exploit this neighbourhood structure.

In [ ]:
dataset = Planetoid(root="/content/pyg_data", name="Cora")
data = dataset[0].to(DEVICE)

print(dataset)
print("nodes:", data.num_nodes)
print("edges:", data.num_edges)
print("node features:", dataset.num_node_features)
print("classes:", dataset.num_classes)
print("train/val/test:", int(data.train_mask.sum()), int(data.val_mask.sum()), int(data.test_mask.sum()))

In [ ]:
class FeatureOnlyMLP(nn.Module):
    def __init__(self, in_channels, hidden_channels, out_channels, dropout=0.5):
        super().__init__()
        self.lin1 = nn.Linear(in_channels, hidden_channels)
        self.lin2 = nn.Linear(hidden_channels, out_channels)
        self.dropout = dropout

    def forward(self, x, edge_index=None):
        x = self.lin1(x)
        x = F.relu(x)
        x = F.dropout(x, p=self.dropout, training=self.training)
        return self.lin2(x)


class SimpleGCN(nn.Module):
    def __init__(self, in_channels, hidden_channels, out_channels, dropout=0.5):
        super().__init__()
        self.conv1 = GCNConv(in_channels, hidden_channels)
        self.conv2 = GCNConv(hidden_channels, out_channels)
        self.dropout = dropout

    def forward(self, x, edge_index):
        x = self.conv1(x, edge_index)
        x = F.relu(x)
        x = F.dropout(x, p=self.dropout, training=self.training)
        return self.conv2(x, edge_index)

In [ ]:
@torch.no_grad()
def evaluate(model, data):
    model.eval()
    logits = model(data.x, data.edge_index)
    pred = logits.argmax(dim=-1)
    result = {}
    for split, mask in [
        ("train", data.train_mask),
        ("val", data.val_mask),
        ("test", data.test_mask),
    ]:
        acc = (pred[mask] == data.y[mask]).float().mean().item()
        result[f"{split}_acc"] = acc
    return result


def train_model(model, data, epochs=200, lr=0.01, weight_decay=5e-4):
    model = model.to(DEVICE)
    optimizer = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=weight_decay)
    history = []
    for epoch in range(1, epochs + 1):
        model.train()
        optimizer.zero_grad()
        logits = model(data.x, data.edge_index)
        loss = F.cross_entropy(logits[data.train_mask], data.y[data.train_mask])
        loss.backward()
        optimizer.step()

        metrics = evaluate(model, data)
        metrics["epoch"] = epoch
        metrics["loss"] = loss.item()
        history.append(metrics)
    return model, pd.DataFrame(history)

In [ ]:
torch.manual_seed(SEED)
mlp = FeatureOnlyMLP(dataset.num_node_features, hidden_channels=64, out_channels=dataset.num_classes)
mlp, hist_mlp = train_model(mlp, data, epochs=200)

torch.manual_seed(SEED)
gcn = SimpleGCN(dataset.num_node_features, hidden_channels=64, out_channels=dataset.num_classes)
gcn, hist_gcn = train_model(gcn, data, epochs=200)

summary = pd.DataFrame([
    {"model": "Feature-only MLP", **evaluate(mlp, data)},
    {"model": "GCN with neighbourhood", **evaluate(gcn, data)},
])
display(summary.round(3))

In [ ]:
def plot_training_curves(hist_mlp, hist_gcn):
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    axes[0].plot(hist_mlp["epoch"], hist_mlp["loss"], label="Feature-only MLP")
    axes[0].plot(hist_gcn["epoch"], hist_gcn["loss"], label="GCN")
    axes[0].set_title("Training loss")
    axes[0].set_xlabel("epoch")
    axes[0].legend()

    axes[1].plot(hist_mlp["epoch"], hist_mlp["test_acc"], label="Feature-only MLP")
    axes[1].plot(hist_gcn["epoch"], hist_gcn["test_acc"], label="GCN")
    axes[1].set_title("Test accuracy")
    axes[1].set_xlabel("epoch")
    axes[1].set_ylim(0, 1)
    axes[1].legend()
    plt.show()


plot_training_curves(hist_mlp, hist_gcn)

## Inspect One Prediction and Its Neighbourhood

The next cell selects a test node where the GCN prediction is correct and the feature-only MLP is wrong, if such a node exists. Then it draws the node's 2-hop neighbourhood.

In [ ]:
@torch.no_grad()
def predictions(model):
    model.eval()
    return model(data.x, data.edge_index).argmax(dim=-1).cpu()

mlp_pred = predictions(mlp)
gcn_pred = predictions(gcn)
y_true = data.y.cpu()
test_nodes = data.test_mask.cpu().nonzero(as_tuple=False).view(-1)

candidates = [
    int(node) for node in test_nodes
    if mlp_pred[node] != y_true[node] and gcn_pred[node] == y_true[node]
]

if candidates:
    focus_node = candidates[0]
    print("Selected a test node where GCN is correct and MLP is wrong:", focus_node)
else:
    focus_node = int(test_nodes[0])
    print("No MLP-wrong/GCN-correct node found in this run; showing first test node:", focus_node)

print("true class:", int(y_true[focus_node]))
print("MLP prediction:", int(mlp_pred[focus_node]))
print("GCN prediction:", int(gcn_pred[focus_node]))

In [ ]:
def draw_k_hop_neighbourhood(node_id, num_hops=2):
    subset, edge_index_sub, mapping, edge_mask = k_hop_subgraph(
        node_id,
        num_hops=num_hops,
        edge_index=data.edge_index.cpu(),
        relabel_nodes=True,
    )
    edges = [tuple(edge) for edge in edge_index_sub.t().tolist()]
    G = nx.Graph()
    G.add_edges_from(edges)
    if not G.nodes:
        G.add_node(0)
    pos = nx.spring_layout(G, seed=SEED)

    colors = []
    for local_node in G.nodes:
        original = int(subset[local_node]) if local_node < len(subset) else node_id
        if original == node_id:
            colors.append("#d62728")
        elif int(y_true[original]) == int(y_true[node_id]):
            colors.append("#2ca02c")
        else:
            colors.append("#cccccc")

    labels = {local: str(int(subset[local])) for local in G.nodes if local < len(subset)}
    plt.figure(figsize=(8, 6))
    nx.draw_networkx_edges(G, pos, alpha=0.35)
    nx.draw_networkx_nodes(G, pos, node_color=colors, node_size=260, edgecolors="#222")
    nx.draw_networkx_labels(G, pos, labels=labels, font_size=8)
    plt.title(f"{num_hops}-hop neighbourhood around node {node_id}\\nred = focus node, green = same class")
    plt.axis("off")
    plt.show()


draw_k_hop_neighbourhood(focus_node, num_hops=2)

## Teaching Takeaway

A standard MLP treats every node independently after reading its feature vector. It can learn from features of a paper, but it cannot directly use who cites whom.

A GNN uses the same node features **and** the graph. In citation data, connected papers often share topics. In spatial data, neighbouring cells, nearby street images, adjacent road segments, or spatially connected observations often carry related information. That is why the GNN comparison is a useful bridge to geospatial AI.